# torch-endid Benchmark: CPU vs GPU

Comparing bootstrap inference performance with GPU acceleration.

In [ ]:
import torch
import time
import numpy as np
import pandas as pd

print(f"PyTorch {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## 1. Simulated Common-Timing Panel

Benchmark endid() with increasing nboot values on a simple DGP.

In [ ]:
from torch_endid import endid

def make_panel(n_units=50, n_periods=10, n_treated=None, tpost1=6, att=2.0, seed=42):
    """Create a simple common-timing panel."""
    if n_treated is None:
        n_treated = n_units // 2
    np.random.seed(seed)
    rows = []
    for i in range(n_units):
        unit_fe = np.random.normal(0, 1)
        for t in range(1, n_periods + 1):
            is_treated = i < n_treated
            is_post = t >= tpost1
            y = unit_fe + 0.5 * np.random.normal()
            if is_treated and is_post:
                y += att
            rows.append({"unit": i, "time": t, "y": y,
                        "post": int(is_post), "D": int(is_treated)})
    return pd.DataFrame(rows)

In [ ]:
# Test across panel sizes to show where GPU starts to help
# GPU overhead dominates for small cross-sections (<500 units)
panel_sizes = [100, 500, 1000, 2000]
nboot = 20

results = {"n_units": [], "cpu_time": [], "gpu_time": [], "speedup": [],
           "cpu_att": [], "gpu_att": []}

for n_units in panel_sizes:
    panel = make_panel(n_units=n_units)
    print(f"\n--- n_units = {n_units} (cross-section ~ {n_units}) ---")
    
    # CPU
    t0 = time.time()
    res_cpu = endid(
        panel, y="y", ivar="unit", tvar="time",
        post="post", dvar="D",
        num_epochs=200, nboot=nboot, nsample=100,
        batch_bootstrap=False, device="cpu",
        verbose=False, seed=42,
    )
    cpu_time = time.time() - t0
    print(f"  CPU: {cpu_time:.1f}s, ATT={res_cpu.att:.3f}")
    
    # GPU (batched)
    torch.cuda.synchronize()
    t0 = time.time()
    res_gpu = endid(
        panel, y="y", ivar="unit", tvar="time",
        post="post", dvar="D",
        num_epochs=200, nboot=nboot, nsample=100,
        batch_bootstrap=True, max_concurrent=4, device="cuda",
        verbose=False, seed=42,
    )
    torch.cuda.synchronize()
    gpu_time = time.time() - t0
    print(f"  GPU: {gpu_time:.1f}s, ATT={res_gpu.att:.3f}")
    print(f"  Speedup: {cpu_time / gpu_time:.2f}x")
    
    results["n_units"].append(n_units)
    results["cpu_time"].append(cpu_time)
    results["gpu_time"].append(gpu_time)
    results["speedup"].append(cpu_time / gpu_time)
    results["cpu_att"].append(res_cpu.att)
    results["gpu_att"].append(res_gpu.att)

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

ax = axes[0]
ax.plot(results["n_units"], results["cpu_time"], "o-", label="CPU", color="tab:blue")
ax.plot(results["n_units"], results["gpu_time"], "s-", label="GPU (batched)", color="tab:orange")
ax.set_xlabel("Number of units (cross-section size)")
ax.set_ylabel("Total time (seconds)")
ax.set_title(f"endid() Runtime: CPU vs GPU (nboot={nboot})")
ax.legend()
ax.grid(True, alpha=0.3)

ax = axes[1]
ax.plot(results["n_units"], results["speedup"], "D-", color="tab:green")
ax.axhline(y=1, color="gray", linestyle="--", alpha=0.5, label="1x (break-even)")
ax.set_xlabel("Number of units (cross-section size)")
ax.set_ylabel("Speedup (CPU / GPU)")
ax.set_title("GPU Speedup Factor")
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("benchmark_endid.png", dpi=150, bbox_inches="tight")
plt.show()

## 2. QTE Visualization

Show the quantile treatment effects from the GPU model.

In [ ]:
# Use the last GPU result
res_gpu.summary()
res_gpu.plot_qte()

## 3. Summary Table

In [ ]:
print(f"{'n_units':>8} {'CPU (s)':>10} {'GPU (s)':>10} {'Speedup':>10} {'CPU ATT':>10} {'GPU ATT':>10}")
print("-" * 65)
for i in range(len(results["n_units"])):
    print(f"{results['n_units'][i]:>8} {results['cpu_time'][i]:>10.1f} "
          f"{results['gpu_time'][i]:>10.1f} {results['speedup'][i]:>9.2f}x "
          f"{results['cpu_att'][i]:>10.3f} {results['gpu_att'][i]:>10.3f}")

print("\nNote: GPU advantage appears at ~500+ units. For typical empirical DiD")
print("panels (<200 units), CPU is faster due to GPU kernel launch overhead.")